In [40]:
# Create Dataset (RAG)
#---------------------------------
financial_rules = """
Savings should be at least 20% of income.
EMI should not exceed 30% of income.
DTI ratio should be less than 60%.
Emergency fund should cover 3–6 months of expenses.
High credit score reduces financial risk.
High EMI + low savings increases risk.
Low savings indicates poor financial discipline.
High income with balanced expenses improves stability.
Loan eligibility improves with low DTI ratio.
Avoid over-leveraging with multiple loans.
"""

with open("financial_rules.txt", "w") as f:
    f.write(financial_rules)

print("Dataset created ✅")



Dataset created ✅


In [41]:
# RAG Setup
# ---------------------------------
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

loader = TextLoader("financial_rules.txt")
docs = loader.load()

splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embeddings)

retriever = vectorstore.as_retriever()

def query_policies(question: str) -> str:
    docs = retriever.invoke(question)
    return "\n".join([doc.page_content for doc in docs])

print("RAG setup complete ✅")

C:\Users\User\AppData\Local\Temp\ipykernel_13712\2605393759.py:14: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2222.04it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG setup complete ✅


In [42]:
# State model
# ---------------------------------
from pydantic import BaseModel
from typing import Optional

class FinancialState(BaseModel):
    user_input: str

    income: Optional[int] = None
    expenses: Optional[int] = None
    existing_emi: Optional[int] = None
    savings: Optional[int] = None
    loan_amount: Optional[int] = None

    retrieved_rules: Optional[str] = None

    financial_health: Optional[str] = None
    risk_score: Optional[float] = None

    emi_ratio: Optional[float] = None
    savings_ratio: Optional[float] = None

    advisory_plan: Optional[str] = None
    final_report: Optional[str] = None

print("State model ready ✅")

State model ready ✅


In [43]:
# TOOL Definition
# ---------------------------------
def financial_metrics_tool(income: float, emi: float, savings: float):
    savings_ratio = savings / income
    emi_ratio = emi / income
    return savings_ratio, emi_ratio

print("Tool ready ✅")

Tool ready ✅


In [44]:
# Extract Profile
# ---------------------------------
import re

def extract_financial_profile(state: FinancialState):
    text = state.user_input.lower()

    # Extract income
    income_match = re.search(r"(earn|income)\D*(\d+)", text)
    if income_match:
        state.income = int(income_match.group(2)) * 1000

    # Extract expenses
    expenses_match = re.search(r"expenses?\D*(\d+)", text)
    if expenses_match:
        state.expenses = int(expenses_match.group(1)) * 1000

    # Extract EMI
    emi_match = re.search(r"emi\D*(\d+)", text)
    if emi_match:
        state.existing_emi = int(emi_match.group(1)) * 1000

    # Extract savings
    savings_match = re.search(r"savings?\D*(\d+)", text)
    if savings_match:
        state.savings = int(savings_match.group(1)) * 1000

    # Extract loan (lakh support)
    loan_match = re.search(r"(\d+)\s*lakh", text)
    if loan_match:
        state.loan_amount = int(loan_match.group(1)) * 100000

    return state

In [45]:
# Retrieve Rules
# ---------------------------------
def retrieve_financial_rules(state: FinancialState):
    state.retrieved_rules = query_policies(state.user_input)
    return state

In [46]:
# Evaluate Health
# ---------------------------------
def evaluate_financial_health(state: FinancialState):

    if not state.income or not state.expenses or not state.existing_emi:
        raise ValueError("Missing financial data. Please provide income, expenses, and EMI.")

    dti = (state.existing_emi + state.expenses) / state.income

    if dti < 0.4:
        state.financial_health = "Good"
        state.risk_score = 30
    elif dti < 0.6:
        state.financial_health = "Moderate"
        state.risk_score = 65
    else:
        state.financial_health = "Risky"
        state.risk_score = 90

    return state

In [47]:
# Tool Node
# ---------------------------------
def calculate_metrics(state: FinancialState):
    savings_ratio, emi_ratio = financial_metrics_tool(
        state.income,
        state.existing_emi,
        state.savings
    )

    state.savings_ratio = savings_ratio
    state.emi_ratio = emi_ratio

    return state

In [48]:
# Advisory Plan
# ---------------------------------
def generate_advisory_plan(state: FinancialState):
    advice = []

    if state.savings_ratio < 0.2:
        advice.append("Increase savings to at least 20% of income")

    if state.emi_ratio > 0.3:
        advice.append("Reduce EMI burden")
    if state.risk_score > 70:
        advice.append("Avoid taking new loans")

    if not advice:
        advice.append("Financial condition is stable")

    state.advisory_plan = "\n".join(advice)

    return state

In [49]:
# Final Report
# ---------------------------------
def final_report(state: FinancialState):
    report = f"""
Financial Health: {state.financial_health}
Risk Score: {state.risk_score}%

Metrics:
Savings Ratio: {round(state.savings_ratio*100,2)}%
EMI Ratio: {round(state.emi_ratio*100,2)}%

Advice:
{state.advisory_plan}
"""

    state.final_report = report
    return state

In [50]:
# Build LangGraph Workflow
# ---------------------------------
from langgraph.graph import StateGraph, END

workflow = StateGraph(FinancialState)

workflow.add_node("extract_financial_profile", extract_financial_profile)
workflow.add_node("retrieve_financial_rules", retrieve_financial_rules)
workflow.add_node("evaluate_financial_health", evaluate_financial_health)
workflow.add_node("calculate_metrics", calculate_metrics)
workflow.add_node("generate_advisory_plan", generate_advisory_plan)
workflow.add_node("final_report", final_report)

workflow.add_edge("extract_financial_profile", "retrieve_financial_rules")
workflow.add_edge("retrieve_financial_rules", "evaluate_financial_health")
workflow.add_edge("evaluate_financial_health", "calculate_metrics")
workflow.add_edge("calculate_metrics", "generate_advisory_plan")
workflow.add_edge("generate_advisory_plan", "final_report")
workflow.add_edge("final_report", END)

workflow.set_entry_point("extract_financial_profile")

app = workflow.compile()

print("Workflow ready ✅")

Workflow ready ✅


In [51]:
# Create Query Function
# ---------------------------------
def run_financial_advisor(query: str):
    """
    Wrapper function to invoke the LangGraph workflow
    """
    result = app.invoke({"user_input": query})
    return result["final_report"]

print("Query function ready ✅")

Query function ready ✅


In [52]:
# Take Input from Terminal
# ---------------------------------

if __name__ == "__main__":
    print("\n💰 Financial Advisory Assistant (Type 'exit' to quit)\n")

    while True:
        user_query = input("Enter your financial details: ")

        if user_query.lower() == "exit":
            print("Goodbye 👋")
            break

        try:
            response = run_financial_advisor(user_query)
            print("\n📊 Advisory Report:\n")
            print(response)
            print("-" * 50)

        except Exception as e:
            print("❌ Error:", e)


💰 Financial Advisory Assistant (Type 'exit' to quit)


📊 Advisory Report:


Financial Health: Risky
Risk Score: 90.0%

Metrics:
Savings Ratio: 12.5%
EMI Ratio: 25.0%

Advice:
Increase savings to at least 20% of income
Avoid taking new loans

--------------------------------------------------
Goodbye 👋


In [ ]:
#I earn 80k, expenses 30k, EMI 20k, savings 10k, want loan 10 lakh